# Dry Eye Disease Prediction - Complete Code
This script contains all code from the dry_eye_disease.ipynb notebook
Best Model: Random Forest with 77.89% accuracy

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/Dry_Eye_Dataset_normalized.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset not found at {DATA_PATH}')

RAW_OUTPUT_PATH = DATA_PATH.with_name('Dry_Eye_Dataset_features.csv')


### Load dataset with optimized settings

In [ ]:
print("Loading dataset...")
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"✅ Loaded dataset with shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()


### Check dataset info

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values:\n{df.isnull().sum().sum()} total missing values")
print(f"\nTarget distribution:\n{df['Dry Eye Disease'].value_counts()}")


### Create a copy for feature engineering

In [ ]:
df_features = df.copy()
print(f"Starting feature extraction. Original features: {df_features.shape[1]}")


In [ ]:
# 1. Blood Pressure Features
if 'Blood pressure_systolic' in df_features.columns and 'Blood pressure_diastolic' in df_features.columns:
    df_features['BP_difference'] = df_features['Blood pressure_systolic'] - df_features['Blood pressure_diastolic']
    df_features['BP_mean'] = (df_features['Blood pressure_systolic'] + df_features['Blood pressure_diastolic']) / 2
    df_features['BP_ratio'] = df_features['Blood pressure_systolic'] / (df_features['Blood pressure_diastolic'] + 1e-6)
    print("✅ Created BP features: BP_difference, BP_mean, BP_ratio")


In [ ]:
# 2. Sleep Quality Features
if 'Sleep duration' in df_features.columns and 'Sleep quality' in df_features.columns:
    df_features['Sleep_efficiency'] = df_features['Sleep duration'] * df_features['Sleep quality']
    df_features['Sleep_deficit'] = 1 - df_features['Sleep duration']  # Inverse of normalized sleep duration
    print("✅ Created sleep features: Sleep_efficiency, Sleep_deficit")

if 'Wake up during night' in df_features.columns and 'Sleep duration' in df_features.columns:
    df_features['Sleep_interruption_score'] = df_features['Wake up during night'] / (df_features['Sleep duration'] + 1e-6)
    print("✅ Created Sleep_interruption_score")


In [ ]:
# 3. Eye Symptom Composite Features
eye_symptoms = ['Discomfort Eye-strain', 'Redness in eye', 'Itchiness/Irritation in eye']
if all(col in df_features.columns for col in eye_symptoms):
    df_features['Eye_symptom_sum'] = df_features[eye_symptoms].sum(axis=1)
    df_features['Eye_symptom_mean'] = df_features[eye_symptoms].mean(axis=1)
    df_features['Eye_symptom_max'] = df_features[eye_symptoms].max(axis=1)
    df_features['Eye_symptom_severity'] = (df_features['Eye_symptom_sum'] / len(eye_symptoms))
    print("✅ Created eye symptom features: Eye_symptom_sum, Eye_symptom_mean, Eye_symptom_max, Eye_symptom_severity")


In [ ]:
# 4. Screen Time and Digital Device Features
if 'Average screen time' in df_features.columns and 'Smart device before bed' in df_features.columns:
    df_features['Screen_exposure'] = df_features['Average screen time'] * (df_features['Smart device before bed'] + 0.5)
    df_features['Digital_strain'] = df_features['Average screen time'] * (1 - df_features.get('Blue-light filter', 0.5))
    print("✅ Created screen features: Screen_exposure, Digital_strain")

if 'Blue-light filter' in df_features.columns and 'Average screen time' in df_features.columns:
    df_features['Unprotected_screen_time'] = df_features['Average screen time'] * (1 - df_features['Blue-light filter'])
    print("✅ Created Unprotected_screen_time")


In [ ]:
# 5. Lifestyle Risk Composite Features
lifestyle_factors = ['Stress level', 'Alcohol consumption', 'Smoking', 'Caffeine consumption']
available_lifestyle = [col for col in lifestyle_factors if col in df_features.columns]

if len(available_lifestyle) > 0:
    df_features['Lifestyle_risk_sum'] = df_features[available_lifestyle].sum(axis=1)
    df_features['Lifestyle_risk_mean'] = df_features[available_lifestyle].mean(axis=1)
    df_features['Lifestyle_risk_score'] = df_features['Lifestyle_risk_mean'] * len(available_lifestyle)
    print(f"✅ Created lifestyle risk features from {available_lifestyle}")


In [ ]:
# 6. Physical Activity and Health Features
if 'Physical activity' in df_features.columns and 'Daily steps' in df_features.columns:
    df_features['Activity_level'] = (df_features['Physical activity'] + df_features['Daily steps']) / 2
    df_features['Sedentary_score'] = 1 - df_features['Activity_level']
    print("✅ Created activity features: Activity_level, Sedentary_score")

if 'Heart rate' in df_features.columns:
    df_features['Heart_rate_normalized'] = df_features['Heart rate']  # Already normalized
    if 'Stress level' in df_features.columns:
        df_features['Cardiac_stress'] = df_features['Heart rate'] * df_features['Stress level']
        print("✅ Created Cardiac_stress")


In [ ]:
# 7. Sleep Disorder and Daytime Sleepiness Features
if 'Sleep disorder' in df_features.columns and 'Feel sleepy during day' in df_features.columns:
    df_features['Sleep_problem_score'] = (df_features['Sleep disorder'] + df_features['Feel sleepy during day']) / 2
    print("✅ Created Sleep_problem_score")

if 'Sleep disorder' in df_features.columns and 'Wake up during night' in df_features.columns:
    df_features['Sleep_disturbance'] = df_features['Sleep disorder'] * df_features['Wake up during night']
    print("✅ Created Sleep_disturbance")


In [ ]:
# 8. Medical Condition Features
if 'Medical issue' in df_features.columns and 'Ongoing medication' in df_features.columns:
    df_features['Medical_complexity'] = (df_features['Medical issue'] + df_features['Ongoing medication']) / 2
    df_features['Health_risk'] = df_features['Medical issue'] * df_features['Ongoing medication']
    print("✅ Created medical features: Medical_complexity, Health_risk")


In [ ]:
# 9. Age-related Risk Features
if 'Age' in df_features.columns:
    # Age groups (since data is normalized, we create risk categories)
    df_features['Age_risk'] = df_features['Age']  # Higher normalized age = higher risk
    if 'Medical issue' in df_features.columns:
        df_features['Age_health_interaction'] = df_features['Age'] * df_features['Medical issue']
        print("✅ Created Age_health_interaction")


In [ ]:
# 10. Overall Dry Eye Risk Score (Composite)
risk_factors = []
if 'Eye_symptom_severity' in df_features.columns:
    risk_factors.append('Eye_symptom_severity')
if 'Screen_exposure' in df_features.columns:
    risk_factors.append('Screen_exposure')
if 'Sleep_problem_score' in df_features.columns:
    risk_factors.append('Sleep_problem_score')
if 'Lifestyle_risk_score' in df_features.columns:
    risk_factors.append('Lifestyle_risk_score')
if 'Medical_complexity' in df_features.columns:
    risk_factors.append('Medical_complexity')

if len(risk_factors) > 0:
    df_features['Dry_eye_risk_score'] = df_features[risk_factors].mean(axis=1)
    print(f"✅ Created composite Dry_eye_risk_score from {len(risk_factors)} factors")


### Summary of feature extraction

In [ ]:
print(f"\n{'='*60}")
print(f"Feature Extraction Complete!")
print(f"{'='*60}")
print(f"Original features: {df.shape[1]}")
print(f"New features created: {df_features.shape[1] - df.shape[1]}")
print(f"Total features: {df_features.shape[1]}")
print(f"Dataset shape: {df_features.shape}")
print(f"\nNew features:")
new_features = [col for col in df_features.columns if col not in df.columns]
for i, feat in enumerate(new_features, 1):
    print(f"  {i}. {feat}")


### Display sample of new features

In [ ]:
print("\nSample of new features:")
display_cols = new_features[:10] + ['Dry Eye Disease']  # Show first 10 new features + target
if len(display_cols) > 1:
    df_features[display_cols].head(10)


### Save the feature-engineered dataset

In [ ]:
df_features.to_csv(RAW_OUTPUT_PATH, index=False)
print(f"\n✅ Saved feature-engineered dataset to: {RAW_OUTPUT_PATH}")
print(f"   Shape: {df_features.shape}")
print(f"   Features: {df_features.shape[1]}")


### Load the feature-engineered dataset

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve
import joblib

FEATURES_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/Dry_Eye_Dataset_features.csv')
print("Loading feature-engineered dataset...")
df_model = pd.read_csv(FEATURES_PATH, low_memory=False)
print(f"✅ Dataset loaded: {df_model.shape}")
print(f"Features: {df_model.shape[1] - 1}, Samples: {df_model.shape[0]}")

# Check target distribution immediately
TARGET_COL = 'Dry Eye Disease'
if TARGET_COL in df_model.columns:
    print(f"\n⚠️ CRITICAL: Checking target distribution...")
    target_dist = df_model[TARGET_COL].value_counts()
    print(f"Target distribution:\n{target_dist}")
    print(f"Unique target values: {sorted(df_model[TARGET_COL].unique())}")
    
    if len(df_model[TARGET_COL].unique()) == 1:
        print(f"\n❌ ERROR: Dataset only contains ONE class!")
        print(f"This is not suitable for binary classification.")
        print(f"Please check the original dataset or use a balanced dataset.")
    elif len(df_model[TARGET_COL].unique()) == 2:
        print(f"✅ Dataset contains both classes - good for binary classification")
    else:
        print(f"⚠️ Dataset contains {len(df_model[TARGET_COL].unique())} classes - may need multi-class approach")


### Prepare features and target

In [ ]:
TARGET_COL = 'Dry Eye Disease'
X = df_model.drop(columns=[TARGET_COL])
y = df_model[TARGET_COL]

print(f"Features shape: {X.shape}")
print(f"\nTarget distribution:\n{y.value_counts()}")
print(f"\nTarget distribution (%):\n{y.value_counts(normalize=True) * 100}")
print(f"\nUnique target values: {sorted(y.unique())}")
print(f"Target dtype: {y.dtype}")

# Check for any missing values
print(f"\nMissing values in X: {X.isnull().sum().sum()}")
print(f"Missing values in y: {y.isnull().sum()}")

# Check if target is binary
if len(y.unique()) != 2:
    print(f"\n⚠️ WARNING: Target has {len(y.unique())} unique values, expected 2 for binary classification!")
    print(f"Unique values: {sorted(y.unique())}")


### Handle any missing values

In [ ]:
if X.isnull().sum().sum() > 0:
    print("Filling missing values with median...")
    X = X.fillna(X.median())
    
if y.isnull().sum() > 0:
    print("Dropping rows with missing target...")
    mask = ~y.isnull()
    X = X[mask]
    y = y[mask]

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n✅ Train set: {X_train.shape[0]} samples")
print(f"✅ Test set: {X_test.shape[0]} samples")
print(f"\nTrain target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")


### Train Random Forest on original feature dataset

In [ ]:
print("\n" + "="*60)
print("RANDOM FOREST - ORIGINAL FEATURE DATASET")
print("="*60)

# Baseline model
rf_baseline = RandomForestClassifier(random_state=42, n_jobs=2, class_weight='balanced')
rf_baseline.fit(X_train, y_train)
baseline_pred = rf_baseline.predict(X_test)
baseline_acc = accuracy_score(y_test, baseline_pred)
print(f"Baseline RF Accuracy: {baseline_acc:.4f} ({baseline_acc*100:.2f}%)")

# Hyperparameter tuning
param_grid_original = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}

print("Starting RandomizedSearchCV for original dataset...")
random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=2),
    param_distributions=param_grid_original,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    n_jobs=2,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
y_pred = best_rf.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)

test_auc = None
if hasattr(best_rf, 'predict_proba') and len(np.unique(y_test)) == 2:
    try:
        y_proba = best_rf.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        test_auc = None

print(f"Best Parameters: {random_search.best_params_}")
print(f"Best CV Score: {random_search.best_score_:.4f} ({random_search.best_score_*100:.2f}%)")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
if test_auc is not None:
    print(f"Test AUC-ROC: {test_auc:.4f} ({test_auc*100:.2f}%)")
else:
    print("Test AUC-ROC: N/A")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


### Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get all possible labels (even if not present in predictions)
# Use np.unique() to handle both numpy arrays and pandas Series
y_test_unique = np.unique(y_test)
y_pred_unique = np.unique(y_pred)
all_labels = sorted(list(set(list(y_test_unique) + list(y_pred_unique))))

if len(all_labels) < 2:
    # If we only have one class, add the missing one for proper matrix shape
    all_labels = [0, 1]

# Create confusion matrix with explicit labels
cm = confusion_matrix(y_test, y_pred, labels=all_labels)

# Handle visualization based on number of classes
if len(all_labels) == 2:
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['No Dry Eye', 'Dry Eye'],
                yticklabels=['No Dry Eye', 'Dry Eye'])
    plt.title('Confusion Matrix - Random Forest')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    print(f"\nConfusion Matrix:")
    if cm.shape == (2, 2):
        print(f"True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
        print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")
    else:
        print(f"⚠️ Only one class present in data")
        print(f"Matrix shape: {cm.shape}")
        print(f"Values: {cm}")
else:
    # Single class case
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix - Random Forest (Single Class)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()
    
    print(f"\n⚠️ WARNING: Only one class found in predictions and test set!")
    print(f"Confusion Matrix (shape {cm.shape}):")
    print(cm)
    print(f"\nThis indicates the dataset may only contain one class (class {all_labels[0]})")


### Feature Importance Analysis

In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': best_rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance.head(20).to_string(index=False))

# Visualize top features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(20)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Important Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Cross-validation score on best model

In [ ]:
print("Performing 5-fold cross-validation on best model...")
cv_scores = cross_val_score(best_rf, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"\nCross-Validation Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
print(f"Std CV Accuracy: {cv_scores.std():.4f} ({cv_scores.std()*100:.2f}%)")
print(f"CV Score Range: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")


### Save the best model

In [ ]:
MODEL_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/best_rf_model_features.pkl')
joblib.dump(best_rf, MODEL_PATH)
print(f"✅ Best Random Forest model saved to: {MODEL_PATH}")

# Save feature importance
FEATURE_IMPORTANCE_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/feature_importance_rf.csv')
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False)
print(f"✅ Feature importance saved to: {FEATURE_IMPORTANCE_PATH}")

print(f"\n{'='*60}")
print(f"MODEL SUMMARY")
print(f"{'='*60}")
print(f"Model Type: Random Forest")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Training Accuracy (CV): {cv_scores.mean():.4f} ({cv_scores.mean()*100:.2f}%)")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
if test_auc is not None:
    print(f"Test AUC-ROC: {test_auc:.4f} ({test_auc*100:.2f}%)")
else:
    print(f"Test AUC-ROC: N/A (only one class predicted)")
print(f"Number of Features: {X.shape[1]}")
print(f"Number of Trees: {best_rf.n_estimators}")


### Load balanced normalized dataset

In [ ]:
BALANCED_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/Dry_Eye_Dataset_normalized_balanced.csv')
print("Loading balanced dataset...")
df_balanced = pd.read_csv(BALANCED_PATH, low_memory=False)
print(f"✅ Balanced dataset loaded: {df_balanced.shape}")

# Check target distribution
print(f"\nTarget distribution:")
print(df_balanced['Dry Eye Disease'].value_counts())
print(f"Target distribution (%):")
print(df_balanced['Dry Eye Disease'].value_counts(normalize=True) * 100)


### Apply the same feature extraction to balanced dataset

In [ ]:
print("Applying feature extraction to balanced dataset...")
df_balanced_features = df_balanced.copy()

# 1. Blood Pressure Features
if 'Blood pressure_systolic' in df_balanced_features.columns and 'Blood pressure_diastolic' in df_balanced_features.columns:
    df_balanced_features['BP_difference'] = df_balanced_features['Blood pressure_systolic'] - df_balanced_features['Blood pressure_diastolic']
    df_balanced_features['BP_mean'] = (df_balanced_features['Blood pressure_systolic'] + df_balanced_features['Blood pressure_diastolic']) / 2
    df_balanced_features['BP_ratio'] = df_balanced_features['Blood pressure_systolic'] / (df_balanced_features['Blood pressure_diastolic'] + 1e-6)

# 2. Sleep Quality Features
if 'Sleep duration' in df_balanced_features.columns and 'Sleep quality' in df_balanced_features.columns:
    df_balanced_features['Sleep_efficiency'] = df_balanced_features['Sleep duration'] * df_balanced_features['Sleep quality']
    df_balanced_features['Sleep_deficit'] = 1 - df_balanced_features['Sleep duration']

if 'Wake up during night' in df_balanced_features.columns and 'Sleep duration' in df_balanced_features.columns:
    df_balanced_features['Sleep_interruption_score'] = df_balanced_features['Wake up during night'] / (df_balanced_features['Sleep duration'] + 1e-6)

# 3. Eye Symptom Composite Features
eye_symptoms = ['Discomfort Eye-strain', 'Redness in eye', 'Itchiness/Irritation in eye']
if all(col in df_balanced_features.columns for col in eye_symptoms):
    df_balanced_features['Eye_symptom_sum'] = df_balanced_features[eye_symptoms].sum(axis=1)
    df_balanced_features['Eye_symptom_mean'] = df_balanced_features[eye_symptoms].mean(axis=1)
    df_balanced_features['Eye_symptom_max'] = df_balanced_features[eye_symptoms].max(axis=1)
    df_balanced_features['Eye_symptom_severity'] = (df_balanced_features['Eye_symptom_sum'] / len(eye_symptoms))

# 4. Screen Time Features
if 'Average screen time' in df_balanced_features.columns and 'Smart device before bed' in df_balanced_features.columns:
    df_balanced_features['Screen_exposure'] = df_balanced_features['Average screen time'] * (df_balanced_features['Smart device before bed'] + 0.5)
    df_balanced_features['Digital_strain'] = df_balanced_features['Average screen time'] * (1 - df_balanced_features.get('Blue-light filter', 0.5))

if 'Blue-light filter' in df_balanced_features.columns and 'Average screen time' in df_balanced_features.columns:
    df_balanced_features['Unprotected_screen_time'] = df_balanced_features['Average screen time'] * (1 - df_balanced_features['Blue-light filter'])

# 5. Lifestyle Risk Features
lifestyle_factors = ['Stress level', 'Alcohol consumption', 'Smoking', 'Caffeine consumption']
available_lifestyle = [col for col in lifestyle_factors if col in df_balanced_features.columns]
if len(available_lifestyle) > 0:
    df_balanced_features['Lifestyle_risk_sum'] = df_balanced_features[available_lifestyle].sum(axis=1)
    df_balanced_features['Lifestyle_risk_mean'] = df_balanced_features[available_lifestyle].mean(axis=1)
    df_balanced_features['Lifestyle_risk_score'] = df_balanced_features['Lifestyle_risk_mean'] * len(available_lifestyle)

# 6. Physical Activity Features
if 'Physical activity' in df_balanced_features.columns and 'Daily steps' in df_balanced_features.columns:
    df_balanced_features['Activity_level'] = (df_balanced_features['Physical activity'] + df_balanced_features['Daily steps']) / 2
    df_balanced_features['Sedentary_score'] = 1 - df_balanced_features['Activity_level']

if 'Heart rate' in df_balanced_features.columns and 'Stress level' in df_balanced_features.columns:
    df_balanced_features['Cardiac_stress'] = df_balanced_features['Heart rate'] * df_balanced_features['Stress level']

# 7. Sleep Disorder Features
if 'Sleep disorder' in df_balanced_features.columns and 'Feel sleepy during day' in df_balanced_features.columns:
    df_balanced_features['Sleep_problem_score'] = (df_balanced_features['Sleep disorder'] + df_balanced_features['Feel sleepy during day']) / 2

if 'Sleep disorder' in df_balanced_features.columns and 'Wake up during night' in df_balanced_features.columns:
    df_balanced_features['Sleep_disturbance'] = df_balanced_features['Sleep disorder'] * df_balanced_features['Wake up during night']

# 8. Medical Condition Features
if 'Medical issue' in df_balanced_features.columns and 'Ongoing medication' in df_balanced_features.columns:
    df_balanced_features['Medical_complexity'] = (df_balanced_features['Medical issue'] + df_balanced_features['Ongoing medication']) / 2
    df_balanced_features['Health_risk'] = df_balanced_features['Medical issue'] * df_balanced_features['Ongoing medication']

# 9. Age-related Features
if 'Age' in df_balanced_features.columns:
    df_balanced_features['Age_risk'] = df_balanced_features['Age']
    if 'Medical issue' in df_balanced_features.columns:
        df_balanced_features['Age_health_interaction'] = df_balanced_features['Age'] * df_balanced_features['Medical issue']

# 10. Overall Dry Eye Risk Score
risk_factors = []
if 'Eye_symptom_severity' in df_balanced_features.columns:
    risk_factors.append('Eye_symptom_severity')
if 'Screen_exposure' in df_balanced_features.columns:
    risk_factors.append('Screen_exposure')
if 'Sleep_problem_score' in df_balanced_features.columns:
    risk_factors.append('Sleep_problem_score')
if 'Lifestyle_risk_score' in df_balanced_features.columns:
    risk_factors.append('Lifestyle_risk_score')
if 'Medical_complexity' in df_balanced_features.columns:
    risk_factors.append('Medical_complexity')

if len(risk_factors) > 0:
    df_balanced_features['Dry_eye_risk_score'] = df_balanced_features[risk_factors].mean(axis=1)

print(f"✅ Feature extraction complete!")
print(f"Original features: {df_balanced.shape[1]}")
print(f"New features: {df_balanced_features.shape[1] - df_balanced.shape[1]}")
print(f"Total features: {df_balanced_features.shape[1]}")


### Prepare features and target for Random Forest

In [ ]:
TARGET_COL = 'Dry Eye Disease'
X_bal = df_balanced_features.drop(columns=[TARGET_COL])
y_bal = df_balanced_features[TARGET_COL]

print(f"Features shape: {X_bal.shape}")
print(f"\nTarget distribution:")
print(y_bal.value_counts())
print(f"\nTarget distribution (%):")
print(y_bal.value_counts(normalize=True) * 100)

# Handle missing values
if X_bal.isnull().sum().sum() > 0:
    X_bal = X_bal.fillna(X_bal.median())
    
if y_bal.isnull().sum() > 0:
    mask = ~y_bal.isnull()
    X_bal = X_bal[mask]
    y_bal = y_bal[mask]

# Train-test split with stratification
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

print(f"\n✅ Train set: {X_train_bal.shape[0]} samples")
print(f"✅ Test set: {X_test_bal.shape[0]} samples")
print(f"\nTrain target distribution:")
print(y_train_bal.value_counts())
print(f"\nTest target distribution:")
print(y_test_bal.value_counts())


In [ ]:
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("DISTRIBUTION OF SCALED NUMERICAL FEATURES IN TRAINING DATA")
print("="*60)

# Identify numerical features
numerical_cols = X_train_bal.select_dtypes(include=[np.number]).columns.tolist()
print(f"\nNumber of numerical features: {len(numerical_cols)}")

# Scale the numerical features
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_bal[numerical_cols]),
    columns=numerical_cols,
    index=X_train_bal.index
)

# Statistical summary of scaled features
print("\nStatistical Summary of Scaled Features:")
print("-" * 60)
summary_stats = X_train_scaled.describe().T
summary_stats['std'] = X_train_scaled.std()
print(summary_stats[['mean', 'std', 'min', '25%', '50%', '75%', 'max']].round(4))

# Distribution plots for top 12 numerical features
print("\nCreating distribution plots for top 12 numerical features...")
top_numerical = numerical_cols[:12]  # Top 12 features

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(top_numerical):
    axes[i].hist(X_train_scaled[col], bins=50, alpha=0.7, edgecolor='black')
    axes[i].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[i].set_xlabel('Scaled Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(True, alpha=0.3)
    # Add mean line
    mean_val = X_train_scaled[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    axes[i].legend(fontsize=8)

plt.suptitle('Distribution of Scaled Numerical Features in Training Data', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/home/prasad526/DryEyeDisease/DryEyeDisease/scaled_features_distribution.png', 
            dpi=300, bbox_inches='tight')
print("✅ Saved distribution plot: scaled_features_distribution.png")
plt.show()

# Box plots for scaled features (top 12)
print("\nCreating box plots for scaled features...")
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(top_numerical):
    box_data = X_train_scaled[col]
    axes[i].boxplot(box_data, vert=True, patch_artist=True)
    axes[i].set_title(f'{col}', fontsize=10, fontweight='bold')
    axes[i].set_ylabel('Scaled Value')
    axes[i].grid(True, alpha=0.3, axis='y')
    # Add statistics
    q1, median, q3 = box_data.quantile([0.25, 0.5, 0.75])
    axes[i].text(1.1, median, f'Med: {median:.2f}', fontsize=8, 
                 verticalalignment='center')

plt.suptitle('Box Plots of Scaled Numerical Features in Training Data', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('/home/prasad526/DryEyeDisease/DryEyeDisease/scaled_features_boxplot.png', 
            dpi=300, bbox_inches='tight')
print("✅ Saved box plot: scaled_features_boxplot.png")
plt.show()

# Correlation heatmap of scaled features (top 20)
print("\nCreating correlation heatmap of top 20 scaled features...")
top_20_numerical = numerical_cols[:20]
correlation_matrix = X_train_scaled[top_20_numerical].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            xticklabels=[col[:15] for col in top_20_numerical],
            yticklabels=[col[:15] for col in top_20_numerical])
plt.title('Correlation Matrix of Top 20 Scaled Numerical Features', 
          fontsize=14, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('/home/prasad526/DryEyeDisease/DryEyeDisease/scaled_features_correlation.png', 
            dpi=300, bbox_inches='tight')
print("✅ Saved correlation heatmap: scaled_features_correlation.png")
plt.show()

# Summary statistics
print("\n" + "="*60)
print("SCALED FEATURES SUMMARY")
print("="*60)
print(f"Total numerical features: {len(numerical_cols)}")
print(f"Mean of all scaled features: {X_train_scaled.mean().mean():.6f}")
print(f"Std of all scaled features: {X_train_scaled.std().mean():.6f}")
print(f"Min value across all features: {X_train_scaled.min().min():.4f}")
print(f"Max value across all features: {X_train_scaled.max().max():.4f}")
print(f"\nFeatures with mean close to 0 (well-scaled): {sum(abs(X_train_scaled.mean()) < 0.01)}")
print(f"Features with std close to 1 (well-scaled): {sum(abs(X_train_scaled.std() - 1) < 0.01)}")


### Random Forest with Hyperparameter Tuning on Balanced Dataset

In [ ]:
print("="*60)
print("Random Forest with Hyperparameter Tuning (Balanced Dataset)")
print("="*60)

# Define parameter grid
param_grid_rf = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None],
    'bootstrap': [True, False],
    'class_weight': [None, 'balanced']
}

# Create Random Forest
rf_balanced = RandomForestClassifier(random_state=42, n_jobs=-1, verbose=0)

# Randomized Search - Reduced parallelism to avoid memory issues
print("Starting RandomizedSearchCV (this may take a few minutes)...")
random_search_bal = RandomizedSearchCV(
    estimator=rf_balanced,
    param_distributions=param_grid_rf,
    n_iter=50,
    cv=5,
    scoring='accuracy',
    n_jobs=2,  # Reduced from -1 to 2 to avoid memory issues
    random_state=42,
    verbose=1
)

random_search_bal.fit(X_train_bal, y_train_bal)

print("\n✅ Hyperparameter tuning complete!")
print(f"Best parameters: {random_search_bal.best_params_}")
print(f"Best CV score: {random_search_bal.best_score_:.4f} ({random_search_bal.best_score_*100:.2f}%)")


### Train best model and evaluate

In [ ]:
best_rf_balanced = random_search_bal.best_estimator_
print("Training best Random Forest model...")
best_rf_balanced.fit(X_train_bal, y_train_bal)

# Predictions
y_pred_bal = best_rf_balanced.predict(X_test_bal)
y_pred_proba_bal = best_rf_balanced.predict_proba(X_test_bal)[:, 1]

# Calculate metrics
test_accuracy_bal = accuracy_score(y_test_bal, y_pred_bal)
test_auc_bal = roc_auc_score(y_test_bal, y_pred_proba_bal)

print(f"\n{'='*60}")
print(f"FINAL MODEL PERFORMANCE (Balanced Dataset)")
print(f"{'='*60}")
print(f"Test Accuracy: {test_accuracy_bal:.4f} ({test_accuracy_bal*100:.2f}%)")
print(f"Test AUC-ROC: {test_auc_bal:.4f} ({test_auc_bal*100:.2f}%)")
print(f"\nClassification Report:")
print(classification_report(y_test_bal, y_pred_bal, target_names=['No Dry Eye', 'Dry Eye']))


### Confusion Matrix for Balanced Dataset

In [ ]:
all_labels_bal = sorted(list(set(list(np.unique(y_test_bal)) + list(np.unique(y_pred_bal)))))
cm_bal = confusion_matrix(y_test_bal, y_pred_bal, labels=all_labels_bal)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_bal, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Dry Eye', 'Dry Eye'],
            yticklabels=['No Dry Eye', 'Dry Eye'])
plt.title('Confusion Matrix - Random Forest (Balanced Dataset)')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print(f"\nConfusion Matrix:")
print(f"True Negatives: {cm_bal[0,0]}, False Positives: {cm_bal[0,1]}")
print(f"False Negatives: {cm_bal[1,0]}, True Positives: {cm_bal[1,1]}")


### Feature Importance for Balanced Dataset

In [ ]:
feature_importance_bal = pd.DataFrame({
    'feature': X_bal.columns,
    'importance': best_rf_balanced.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 Most Important Features:")
print(feature_importance_bal.head(20).to_string(index=False))

# Visualize
plt.figure(figsize=(10, 8))
top_features_bal = feature_importance_bal.head(20)
plt.barh(range(len(top_features_bal)), top_features_bal['importance'])
plt.yticks(range(len(top_features_bal)), top_features_bal['feature'])
plt.xlabel('Feature Importance')
plt.title('Top 20 Most Important Features - Random Forest (Balanced)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
cv_scores_bal = cross_val_score(best_rf_balanced, X_train_bal, y_train_bal, cv=5, scoring='accuracy', n_jobs=2)
print(f"Cross-Validation Scores: {cv_scores_bal}")
print(f"Mean CV Accuracy: {cv_scores_bal.mean():.4f} ({cv_scores_bal.mean()*100:.2f}%)")
print(f"Std CV Accuracy: {cv_scores_bal.std():.4f}")

# Save model
MODEL_PATH_BAL = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/best_rf_model_balanced_features.pkl')
joblib.dump(best_rf_balanced, MODEL_PATH_BAL)
print(f"\n✅ Best Random Forest model saved to: {MODEL_PATH_BAL}")

# Save feature importance
FEATURE_IMPORTANCE_PATH_BAL = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/feature_importance_rf_balanced.csv')
feature_importance_bal.to_csv(FEATURE_IMPORTANCE_PATH_BAL, index=False)
print(f"✅ Feature importance saved to: {FEATURE_IMPORTANCE_PATH_BAL}")

print(f"\n{'='*60}")
print(f"FINAL SUMMARY - BALANCED DATASET")
print(f"{'='*60}")
print(f"Model Type: Random Forest")
print(f"Best Parameters: {random_search_bal.best_params_}")
print(f"Training Accuracy (CV): {cv_scores_bal.mean():.4f} ({cv_scores_bal.mean()*100:.2f}%)")
print(f"Test Accuracy: {test_accuracy_bal:.4f} ({test_accuracy_bal*100:.2f}%)")
print(f"Test AUC-ROC: {test_auc_bal:.4f} ({test_auc_bal*100:.2f}%)")
print(f"Number of Features: {X_bal.shape[1]}")
print(f"Number of Trees: {best_rf_balanced.n_estimators}")


### Save the balanced features dataset to Dry_Eye_Dataset_features.csv

In [ ]:
FEATURES_OUTPUT_PATH = Path('/home/prasad526/DryEyeDisease/DryEyeDisease/Dry_Eye_Dataset_features.csv')
df_balanced_features.to_csv(FEATURES_OUTPUT_PATH, index=False)
print(f"✅ Balanced features dataset saved to: {FEATURES_OUTPUT_PATH}")
print(f"   Shape: {df_balanced_features.shape}")
print(f"   Features: {df_balanced_features.shape[1] - 1} (excluding target)")
print(f"   Samples: {df_balanced_features.shape[0]}")
print(f"\n   Target distribution:")
print(df_balanced_features['Dry Eye Disease'].value_counts())
print(f"\n   This file now contains BOTH classes (0 and 1) and can be used for Random Forest training!")


In [ ]:
# 4. Stacking - Meta-learner approach
from sklearn.ensemble import StackingClassifier

print("="*60)
print("STACKING CLASSIFIER - Meta-Learner Approach")
print("="*60)

# Use selected features if available
if 'selected_features' in locals() and len(selected_features) > 0:
    X_train_stack = X_train_bal[selected_features]
    X_test_stack = X_test_bal[selected_features]
else:
    X_train_stack = X_train_bal
    X_test_stack = X_test_bal

# Base models - Reduced parallelism to avoid memory issues
base_models = [
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=2)),
    ('gb', GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=5, random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=200, learning_rate=0.1, random_state=42))
]

# Meta-learner (Logistic Regression)
stacking_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=3,  # Reduced from 5 to 3 to reduce memory usage
    stack_method='predict_proba'
)

print("Training Stacking Classifier...")
stacking_clf.fit(X_train_stack, y_train_bal)

y_pred_stack = stacking_clf.predict(X_test_stack)
y_pred_proba_stack = stacking_clf.predict_proba(X_test_stack)[:, 1]

stack_acc = accuracy_score(y_test_bal, y_pred_stack)
stack_auc = roc_auc_score(y_test_bal, y_pred_proba_stack)

print(f"\n✅ Stacking Classifier Results:")
print(f"   Accuracy: {stack_acc:.4f} ({stack_acc*100:.2f}%)")
print(f"   AUC-ROC: {stack_auc:.4f} ({stack_auc*100:.2f}%)")
print(f"\n   Classification Report:")
print(classification_report(y_test_bal, y_pred_stack, target_names=['No Dry Eye', 'Dry Eye']))


### Best Model Summary

In [ ]:
print("="*60)
print("BEST MODEL PERFORMANCE")
print("="*60)

if 'test_accuracy_bal' in globals() and 'test_auc_bal' in globals():
    print(f"\n🏆 BEST MODEL: Original RF (Balanced)")
    print(f"   Accuracy: {test_accuracy_bal:.4f} ({test_accuracy_bal*100:.2f}%)")
    print(f"   AUC-ROC: {test_auc_bal:.4f} ({test_auc_bal*100:.2f}%)")
    print(f"\n   This model outperformed all other methods tested:")
    print(f"   - RF + Feature Selection: 76.55%")
    print(f"   - Voting Ensemble: 76.78%")
    print(f"   - RF + Extended Tuning: 76.82%")
    print(f"   - Stacking Classifier: 76.41%")
    print(f"   - RF + Outlier Removal: 75.92%")
    print(f"   - RF + Polynomial Features: 76.41%")
    print(f"\n   ✅ Lower-accuracy methods have been removed from the notebook.")
else:
    print("⚠️ Run Random Forest training first (cells 34-35)")